Install Libraries

In [37]:
!pip install -q --upgrade torchao peft trl transformers accelerate datasets bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 124.6 MB/s eta 0:00:00


Setup Environment & Imports

In [38]:
import os
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

# HF Token
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "YOUR_HUGGINGFACE_TOKEN"

# Select light base model
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"✅ Environment initialized. Using device: {device}")

✅ Environment initialized. Using device: cuda


Prepare Dataset

In [39]:
# Sample domain-specific instruction dataset (Medical Support)
data = {
    "instruction": [
        "What are the symptoms of mild seasonal allergies?",
        "How should I handle a minor burn at home?",
        "What is the recommended daily water intake?",
        "What should I do if I have a mild headache?",
        "How do I clean a small superficial cut?"
    ],
    "response": [
        "Common symptoms include sneezing, runny or stuffy nose, itchy eyes, and mild coughing.",
        "Cool the burn with cool running water for 10-15 minutes. Do not apply ice directly. Cover with a clean, non-stick bandage.",
        "An average adult should aim for around 2 to 3 liters (8-12 cups) of water per day depending on activity level.",
        "Rest in a quiet room, stay hydrated, and consider over-the-counter pain relief if appropriate.",
        "Wash your hands, gently rinse the cut with water and mild soap, apply an antiseptic cream, and cover with a bandage."
    ]
}

# Convert to Hugging Face Dataset format
dataset = Dataset.from_dict(data)

def format_prompts(example):
    text = f"<|system|>\nYou are a helpful medical customer support assistant.</s>\n<|user|>\n{example['instruction']}</s>\n<|assistant|>\n{example['response']}</s>"
    return {"text": text}

formatted_dataset = dataset.map(format_prompts)
print("✅ Dataset Prepared Successfully!")

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

✅ Dataset Prepared Successfully!


Test Base Model (Before Fine-Tuning)

In [40]:
# Load Base Model & Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

test_prompt = "<|system|>\nYou are a helpful medical customer support assistant.</s>\n<|user|>\nHow should I handle a minor burn at home?</s>\n<|assistant|>\n"

inputs = tokenizer(test_prompt, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = base_model.generate(**inputs, max_new_tokens=80)

before_tuning_response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("--- 🔴 RESPONSE BEFORE FINE-TUNING ---")
print(before_tuning_response)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- 🔴 RESPONSE BEFORE FINE-TUNING ---
<|system|>
You are a helpful medical customer support assistant.
<|user|>
How should I handle a minor burn at home?
<|assistant|>
If you have a minor burn at home, the first step is to ensure that the burn is not too severe. If it is not too severe, you can try the following steps to help minimize the damage:

1. Remove the burn: If the burn is small and not too deep, you can remove it by applying a cold compress or a wet towel to the affected area.


Fine-Tuning with LoRA

In [41]:
# 1. Configure LoRA
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 2. Set Training Arguments
sft_config = SFTConfig(
    dataset_text_field="text",
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    logging_steps=1,
    learning_rate=2e-4,
    fp16=True,
    save_strategy="no"
)

# 3. Train Model
trainer = SFTTrainer(
    model=base_model,
    train_dataset=formatted_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=sft_config
)

print("🚀 Starting Fine-Tuning Process...")
trainer.train()
print("✅ Fine-Tuning Completed Successfully!")

# 4. Save Fine-Tuned LoRA Adapter
adapter_path = "./fine_tuned_adapter"
trainer.model.save_pretrained(adapter_path)
print(f"💾 Fine-tuned LoRA Adapter saved to {adapter_path}")

Adding EOS to train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


🚀 Starting Fine-Tuning Process...


Step,Training Loss
1,2.518681
2,2.579817
3,2.582821
4,2.616056
5,2.280811
6,2.196520
7,2.113252
8,2.270166
9,1.922203
10,1.944319


✅ Fine-Tuning Completed Successfully!
💾 Fine-tuned LoRA Adapter saved to ./fine_tuned_adapter


Evaluate & Compare Results (After Fine-Tuning)

In [44]:
# 🎯 Cell 6: Evaluate & Compare Results (After Fine-Tuning)

# 1. Switch model to evaluation mode (Crucial fix!)
trainer.model.eval()

# 2. Prepare inputs
inputs = tokenizer(test_prompt, return_tensors="pt").to(device)

# 3. Generate text with appropriate parameters
with torch.no_grad():
    ft_outputs = trainer.model.generate(
        **inputs,
        max_new_tokens=100,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1
    )

after_tuning_response = tokenizer.decode(ft_outputs[0], skip_special_tokens=True)

print("\n==================================================")
print("📊 FINAL RESPONSE COMPARISON")
print("==================================================")
print("🔴 BEFORE FINE-TUNING:\n", before_tuning_response)
print("\n🟢 AFTER FINE-TUNING:\n", after_tuning_response)
print("==================================================")

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📊 FINAL RESPONSE COMPARISON
🔴 BEFORE FINE-TUNING:
 <|system|>
You are a helpful medical customer support assistant.
<|user|>
How should I handle a minor burn at home?
<|assistant|>
If you have a minor burn at home, the first step is to ensure that the burn is not too severe. If it is not too severe, you can try the following steps to help minimize the damage:

1. Remove the burn: If the burn is small and not too deep, you can remove it by applying a cold compress or a wet towel to the affected area.

🟢 AFTER FINE-TUNING:
 <|system|>
You are a helpful medical customer support assistant.
<|user|>
How should I handle a minor burn at home?
<|assistant|>
Here are some steps to follow if you experience a minor burn at home:
1. Turn off the power source and unplug any electrical appliances that may have been involved in the incident, such as the microwave or oven.
2. Remove the affected clothing and any other clothing that may be soiled with the burn.
3. Gently clean up the burned area using